<a href="https://colab.research.google.com/github/infernoWolf99/DL/blob/main/MedNextWithAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install torchio monai fvcore
import os, sys
print('Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.4/188.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!git clone https://github.com/MIC-DKFZ/MedNeXt.git mednext
%cd mednext
!pip -q install -e .
%cd ..

sys.path.append('mednext')

Cloning into 'mednext'...
remote: Enumerating objects: 762, done.
remote: Counting objects: 100% (245/245), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 762 (delta 220), reused 206 (delta 206), pack-reused 517 (from 1)
Receiving objects: 100% (762/762), 539.81 KiB | 12.85 MiB/s, done.
Resolving deltas: 100% (446/446), done.
/content/mednext
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.4 MB/s eta 0:00:00
/content


In [3]:
import torch
import torch.nn as nn
from fvcore.nn import FlopCountAnalysis
# from fvcore.nn import parameter_count_table
import torchio as tio
from torchio.data import SubjectsLoader, SubjectsDataset
import matplotlib.pyplot as plt
from glob import glob
import numpy as np
from tqdm import tqdm
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.losses import DiceLoss
from matplotlib.colors import ListedColormap
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

In [4]:
!mkdir training validation
!unzip -qq /content/drive/MyDrive/Brats-Data/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData.zip -d training
!unzip -qq /content/drive/MyDrive/Brats-Data/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData.zip -d validation

In [5]:
training_path = '/content/training/ASNR-MICCAI-BraTS2023-SSA-Challenge-TrainingData_V2'
validation_path = '/content/validation/ASNR-MICCAI-BraTS2023-SSA-Challenge-ValidationData'

In [6]:
print(len(os.listdir(training_path)))
print(len(os.listdir(validation_path)))

# torch.tensor([[1.0, 2.0, 2.0],
#               [4.0, 5.0, 6.0]]).sum(dim=1)

60
15


In [7]:
class BratsDataset(SubjectsDataset):
    def __init__(self, root_dir: str, transform=None):
        self.patient_dirs = os.listdir(root_dir)
        self.transform = transform
        self.subjects = []

        for patient_dir in self.patient_dirs:
            t1c_path = glob(os.path.join(root_dir, patient_dir, "*t1c.nii.gz*"), recursive=True)
            t1n_path = glob(os.path.join(root_dir, patient_dir, "*t1n.nii.gz*"), recursive=True)
            t2f_path = glob(os.path.join(root_dir, patient_dir, "*t2f.nii.gz*"), recursive=True)
            t2w_path = glob(os.path.join(root_dir, patient_dir, "*t2w.nii.gz*"), recursive=True)
            seg_path = glob(os.path.join(root_dir, patient_dir, "*seg.nii.gz*"), recursive=True)

            if not (t1c_path and t1n_path and t2f_path and t2w_path and seg_path):
                print(f"Skipping {patient_dir}")
                print(t1c_path)
                continue

            self.subjects.append(
                 tio.Subject(
                    vol=tio.ScalarImage([t1c_path[0], t1n_path[0], t2f_path[0], t2w_path[0]]),
                    label=tio.LabelMap(seg_path[0]),
                )
            )

        super().__init__(self.subjects, transform = transform)

training_dataset = BratsDataset(root_dir=training_path)
# validation_dataset = BratsDataset(root_dir=validation_path)

train_set, val_set = torch.utils.data.random_split(training_dataset, [.8, .2])

train_loader = SubjectsLoader(train_set, batch_size=2, shuffle=True, num_workers=2)
val_loader = SubjectsLoader(val_set, batch_size=2, shuffle=False, num_workers=2)


In [8]:
# --- dictinary containing all tunable parameters --- #

parameters = {
    'kernel_size': 7,        # could also be 2, 5
    'group_size': 1,         # group size of normalization layers...can try using values like 2, 4, 6, 8
    'norm_type': 'group',    # type of normalization used...other supported types are 'instance' and 'layer'
    'activation_fn': 'gelu', # other supported types are 'leaky' and 'swish'
    'lr': 1e-3,              # learning rate
    'optimizer': 'adam',     # options are 'adam' and 'adamw'
    'num_epochs': 50,
    'scheduler': 'reduceLROnPlateau', # options are 'reduceLROnPlateau' and 'cosineAnnealing'
    'patience': 5,                    # scheduler patience works only when using reduceLRONPlataeu
    'stopping_patience': 5,           # controls early stopping patience
    'uname': 'seth',                  # username of the person training
}

In [21]:
# --- Squeeze-and-Excitation Block (3D) ---
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock3D, self).__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y


# --- Axial Attention ---
class AxialAttention3D(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        b, c, d, h, w = x.shape
        x = x.permute(0, 2, 3, 4, 1).reshape(b * d * h, w, c)
        qkv = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.reshape(-1, t.shape[1], self.heads, c // self.heads).transpose(1, 2) for t in qkv]

        attn = (q @ k.transpose(-1, -2)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(x.shape[0], x.shape[1], c)
        out = self.proj(out).reshape(b, d, h, w, c).permute(0, 4, 1, 2, 3)
        return out


# --- MedNeXt Block with SE ---
class MedNeXtBlock(nn.Module):
    def __init__(self, channels, kernel_size=parameters['kernel_size'], act_fn=parameters['activation_fn'], norm_type=parameters['norm_type'], group_size=parameters['group_size']):
        super().__init__()
        self.dw_conv = nn.Conv3d(channels, channels, kernel_size, padding=kernel_size // 2, groups=channels)

        if norm_type == 'instance':
            self.norm = nn.InstanceNorm3d(group_size, channels)
        elif norm_type == 'layer':
            self.norm = nn.LayerNorm(group_size, channels)
        else:
            self.norm = nn.GroupNorm(group_size, channels)

        if act_fn == 'gelu':
            self.act_fn = nn.GELU()
        elif act_fn == 'leaky':
            self.act_fn = nn.LeakyReLU()
        elif act_fn == 'swish':
            self.act_fn == nn.SiLU()
        else:
            print('activation function can on only be one of "gelu", "leaky", "swish"')

        self.pw_conv = nn.Conv3d(channels, channels, kernel_size=1)
        self.se = SEBlock3D(channels)

    def forward(self, x):
        out = self.dw_conv(x)
        out = self.norm(out)
        out = self.act_fn(out)
        out = self.pw_conv(out)
        out = self.se(out)
        return out + x  # Residual connection


# --- Full MedNeXt with Attention Model ---
class MedNeXtWithAttention(nn.Module):
    def __init__(self, in_channels=1, out_channels=3, base_channels=32):
        super().__init__()
        self.stem = nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1)

        # Encoder
        self.enc1 = MedNeXtBlock(base_channels)
        self.down1 = nn.Conv3d(base_channels, base_channels * 2, kernel_size=2, stride=2)

        self.enc2 = MedNeXtBlock(base_channels * 2)
        self.down2 = nn.Conv3d(base_channels * 2, base_channels * 4, kernel_size=2, stride=2)

        # Bottleneck with axial attention
        self.bottleneck = nn.Sequential(
            MedNeXtBlock(base_channels * 4),
            AxialAttention3D(base_channels * 4)
        )

        # Decoder
        self.up2 = nn.ConvTranspose3d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = MedNeXtBlock(base_channels * 2)

        self.up1 = nn.ConvTranspose3d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = MedNeXtBlock(base_channels)

        self.final = nn.Conv3d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.stem(x)
        x1 = self.enc1(x1)
        x2 = self.down1(x1)
        x2 = self.enc2(x2)
        x3 = self.down2(x2)

        x3 = self.bottleneck(x3)

        x = self.up2(x3) + x2
        x = self.dec2(x)
        x = self.up1(x) + x1
        x = self.dec1(x)
        return self.final(x)

model = MedNeXtWithAttention(in_channels=4, out_channels=4, base_channels=32)
dummy = torch.randn(1, 4, 64, 64, 64)  # Example 3D input (B, C, D, H, W)
out = model(dummy)
print(out.shape)  # Expected: (1, 4, 64, 64, 64)

torch.Size([1, 4, 64, 64, 64])


In [22]:
class BratsModelTrainer:
    def __init__(self, model, train_loader, val_loader, num_classes=4, device=None, parameters=parameters):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.num_classes = num_classes
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.parameters = parameters

        self.dice_metric = DiceMetric(
            include_background=False,
            reduction="none"
        )
        self.hd95_metric = HausdorffDistanceMetric(
            include_background=False,
            percentile=95,
            reduction="none",
            get_not_nans=False
        )

        self.reset_metrics()

    def reset_metrics(self):
        self.metrics = {
            'train_loss': [],
            'val_loss': [],
            'dice': {f'class_{i}': [] for i in range(1, self.num_classes)},
            'hd95': {f'class_{i}': [] for i in range(1, self.num_classes)},
            'mean_dice': [],
            'mean_hd95': []
        }

    def train(self):
        criterion = DiceLoss(to_onehot_y=True, softmax=True)

        if self.parameters['optimizer'] == 'adam':
            optimizer = torch.optim.Adam(self.model.parameters(), lr=self.parameters['lr'])
        elif self.paramters['optimzer'] == 'adamw':
            optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.parameters['lr'])
        else:
            print('optimzer options can only be adam and adamw')
            return

        if self.parameters['scheduler'] == 'reduceLROnPlateau':
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                                   mode='min',
                                                                   factor=0.1,
                                                                   patience=5,
                                                                   threshold=0.0001,
                                                                   threshold_mode='rel',
                                                                   cooldown=0,
                                                                   min_lr=0,
                                                                   eps=1e-08)
        elif self.parameters['scheduler'] == 'cosineAnnealing':
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                                   T_max=self.parameters['num_epochs'],
                                                                   eta_min=1e-5,
                                                                   last_epoch=-1)
        else:
            print('scheduler can only be "cosineAnnealing or "reduceLROnPlateau"')
            return

        best_val_loss = float('inf')
        epochs_no_improve = 0
        patience = self.parameters['stopping_patience']

        for epoch in tqdm(range(self.parameters['num_epochs']), desc="Training"):

            self.model.train()
            epoch_train_loss = 0.0

            for batch in tqdm(self.train_loader, desc=f"Epoch {epoch+1}", leave=False):
                inputs = batch["vol"][tio.DATA].to(self.device).float()
                labels = batch["label"][tio.DATA].long().to(self.device)
                labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)

                optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                epoch_train_loss += loss.item()

            self.metrics['train_loss'].append(epoch_train_loss / len(self.train_loader))

            val_loss = self.validate()
            self.metrics['val_loss'].append(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                epochs_no_improve = 0
                torch.save(self.model.state_dict(), "best_model.pth")
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    break

            scheduler.step()

        return self.metrics

    def validate(self):
        self.model.eval()
        criterion = DiceLoss(to_onehot_y=True, softmax=True)
        epoch_val_loss = 0.0

        self.dice_metric.reset()
        self.hd95_metric.reset()

        with torch.no_grad():
            for batch in self.val_loader:
                inputs = batch["vol"][tio.DATA].to(self.device).float()
                labels = batch["label"][tio.DATA].long().to(self.device)
                labels = torch.where(labels == 4, torch.tensor(3, device=labels.device), labels)

                outputs = self.model(inputs)
                loss = criterion(outputs, labels)
                epoch_val_loss += loss.item()

                outputs_softmax = torch.softmax(outputs, dim=1)
                labels_onehot = torch.nn.functional.one_hot(labels.squeeze(1), num_classes=self.num_classes).permute(0, 4, 1, 2, 3)

                self.dice_metric(outputs_softmax, labels_onehot)
                self.hd95_metric(outputs_softmax, labels_onehot)

        dice_values = self.dice_metric.aggregate()
        hd95_values = self.hd95_metric.aggregate()

        for class_idx in range(1, self.num_classes):
            class_dice = dice_values[:, class_idx-1].mean().item()
            class_hd95 = hd95_values[:, class_idx-1].mean().item()

            self.metrics['dice'][f'class_{class_idx}'].append(class_dice)
            self.metrics['hd95'][f'class_{class_idx}'].append(class_hd95)

        self.metrics['mean_dice'].append(dice_values.mean().item())
        self.metrics['mean_hd95'].append(hd95_values.mean().item())

        return epoch_val_loss / len(self.val_loader)

    @staticmethod
    def predict(model, input_volume, device=None):
        device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.eval()

        with torch.no_grad():
            if len(input_volume.shape) == 4:
                input_volume = input_volume.unsqueeze(0)
            input_volume = input_volume.to(device).float()
            outputs = model(input_volume)
            preds = torch.argmax(outputs, dim=1)

        preds = preds.squeeze(0).cpu().numpy()

        slice_idx = slice_idx if slice_idx is not None else input_volume.shape[-3] // 2
        input_slice = input_volume[0, slice_idx].cpu().numpy()
        pred_slice = preds[slice_idx]

        BratsModelTrainer.visualize_prediction(
            input_slice=input_slice,
            ground_truth=np.zeros_like(pred_slice),
            prediction=pred_slice,
            # save_path=save_path
        )

    def plot_and_save_loss(self):

        save_path=f"{self.parameters['uname']}/loses.png"
        plt.figure(figsize=(15, 10))

        plt.subplot(2, 2, 1)
        plt.plot(self.metrics['train_loss'], label='Train Loss')
        plt.plot(self.metrics['val_loss'], label='Validation Loss')
        plt.title('Training and Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()

    def save_metrics_and_params(self):
        metrics_path=f"{self.parameters['uname']}/metrics.csv"
        params_path="model_parameters.csv"
        metrics_df = pd.DataFrame(self.metrics)
        metrics_df.to_csv(metrics_path, index=False)
        params_df = pd.DataFrame(self.parameters)
        params_df.to_csv(params_path, index=False)

    @staticmethod
    def visualize_prediction(
        input_slice,
        ground_truth,
        prediction,
        save_path=f"prediction.png",
        alpha=0.5
    ):

        colors = ['black', 'red', 'green', 'blue', 'yellow']
        cmap = ListedColormap(colors[:len(np.unique(ground_truth))])

        plt.figure(figsize=(18, 6))

        plt.subplot(1, 3, 1)
        plt.imshow(input_slice, cmap='gray')
        plt.title('Input Image')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(input_slice, cmap='gray')
        plt.imshow(ground_truth, cmap=cmap, alpha=alpha, vmin=0, vmax=len(colors)-1)
        plt.title('Ground Truth')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(input_slice, cmap='gray')
        plt.imshow(prediction, cmap=cmap, alpha=alpha, vmin=0, vmax=len(colors)-1)
        plt.title('Prediction')
        plt.axis('off')

        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()


In [23]:
brats_model = BratsModelTrainer(model=model,
                                val_loader=val_loader,
                                train_loader=train_loader,
                                )

In [24]:
brats_model.train()

Training:   0%|          | 0/50 [00:08<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.13 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.01 GiB is free. Process 10209 has 13.72 GiB memory in use. Of the allocated memory 13.58 GiB is allocated by PyTorch, and 17.62 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)